# Введение в MapReduce модель на Python


In [286]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [287]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [288]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [289]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [290]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [291]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [292]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [293]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [294]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [295]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [296]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [297]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [298]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [299]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [300]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.7252369690944507)),
 (1, np.float64(1.7252369690944507)),
 (2, np.float64(1.7252369690944507)),
 (3, np.float64(1.7252369690944507)),
 (4, np.float64(1.7252369690944507))]

## Inverted index

In [301]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [302]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [303]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [304]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [])]

## TeraSort

In [305]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.004511271237776726)),
   (None, np.float64(0.02597011190727494)),
   (None, np.float64(0.041906556579744514)),
   (None, np.float64(0.06380878994287975)),
   (None, np.float64(0.07527197424412746)),
   (None, np.float64(0.09314252125713307)),
   (None, np.float64(0.1210522164686948)),
   (None, np.float64(0.15740573426543025)),
   (None, np.float64(0.1781056275133085)),
   (None, np.float64(0.17816717932193504)),
   (None, np.float64(0.2058097353175442)),
   (None, np.float64(0.23564201983590016)),
   (None, np.float64(0.2517289633974199)),
   (None, np.float64(0.2874704623647112)),
   (None, np.float64(0.30535994862277327))]),
 (1,
  [(None, np.float64(0.503587511089455)),
   (None, np.float64(0.5051859145385256)),
   (None, np.float64(0.5587357132400649)),
   (None, np.float64(0.5671005065862599)),
   (None, np.float64(0.6240413785507753)),
   (None, np.float64(0.6534088913284685)),
   (None, np.float64(0.6749243210971086)),
   (None, np.float64(0.71757691

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [306]:
input_numbers = [1, 45, 2, 100, 34, 99, 120, 11]

def RECORDREADER():
    return [(i, val) for i, val in enumerate(input_numbers)]

def MAP(_, value):
    yield ("max", value)

def REDUCE(key, values):
    yield (key, max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('max', 120)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [307]:
def MAP(_, value):
    yield ("avg", (value, 1))

def REDUCE(key, values):
    sum_val = 0
    count = 0
    for val, c in values:
        sum_val += val
        count += c
    yield (key, sum_val / count)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('avg', 51.5)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [308]:
def groupbykey_sort(iterable):
    sorted_iterable = sorted(iterable, key=lambda x: x[0])

    if not sorted_iterable:
        return

    current_key, current_group = sorted_iterable[0][0], []

    for k, v in sorted_iterable:
        if k == current_key:
            current_group.append(v)
        else:
            yield (current_key, current_group)
            current_key, current_group = k, [v]

    yield (current_key, current_group)

# Проверка
map_out = [(25, 'user1'), (33, 'user3'), (25, 'user2')]
print(list(groupbykey_sort(map_out)))

[(25, ['user1', 'user2']), (33, ['user3'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [309]:
data = ["apple", "banana", "apple", "orange", "banana"]

def RECORDREADER():
    return [(i, v) for i, v in enumerate(data)]

def MAP(_, value):
    yield (value, None)

def REDUCE(value, _):
    yield value

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

['apple', 'banana', 'orange']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [310]:
# Данные: Пользователи (id, name, age)
users = [
    (1, "Alice", 25), (2, "Bob", 40), (3, "Charlie", 35), (4, "Diana", 20)
]

def RECORDREADER():
    return [(u[0], u) for u in users]

# Условие: возраст > 30
def MAP(_, row):
    if row[2] > 30:
        yield (row, row)

def REDUCE(key, values):
    yield key

print("Результат Selection (age > 30):")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Результат Selection (age > 30):
[(2, 'Bob', 40), (3, 'Charlie', 35)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [311]:
# Данные: Продажи (id, product, category)
sales = [
    (101, "iPhone", "Electronics"),
    (102, "iPad", "Electronics"),
    (103, "Bread", "Food")
]

def RECORDREADER():
    return [(s[0], s) for s in sales]

# Проекция на атрибут 'category'
def MAP(_, row):
    t_prime = row[2]
    yield (t_prime, t_prime)

def REDUCE(t_prime, values):
    yield t_prime

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

['Electronics', 'Food']


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [312]:
R = [(1, "A"), (2, "B")]
S = [(2, "B"), (3, "C")]

def RECORDREADER():
    combined = [('R', t) for t in R] + [('S', t) for t in S]
    return [(None, t[1]) for t in combined]

def MAP(_, t):
    yield (t, t)

def REDUCE(t, values):
    yield t

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(1, 'A'), (2, 'B'), (3, 'C')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [313]:
def MAP(_, t):
    yield (t, t)

def REDUCE(t, values):
    if len(list(values)) >= 2:
        yield t

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(2, 'B')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [314]:
def RECORDREADER():
    return [(None, (t, 'R')) for t in R] + [(None, (t, 'S')) for t in S]

def MAP(_, val):
    t, rel = val
    yield (t, rel)

def REDUCE(t, rels):
    if list(rels) == ['R']:
        yield t

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(1, 'A')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [315]:
# R(a, b): (Сотрудник, ID_Департамента)
R = [("Alice", 10), ("Bob", 20)]
# S(b, c): (ID_Департамента, Название)
S = [(10, "HR"), (20, "IT")]

def RECORDREADER():
    return [(None, ('R', r)) for r in R] + [(None, ('S', s)) for s in S]

def MAP(_, val):
    rel, data = val
    if rel == 'R':
        yield (data[1], ('R', data[0]))
    else:
        yield (data[0], ('S', data[1]))

def REDUCE(b, values):
    res_r = [v[1] for v in values if v[0] == 'R']
    res_s = [v[1] for v in values if v[0] == 'S']
    for a in res_r:
        for c in res_s:
            yield (a, b, c)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[('Alice', 10, 'HR'), ('Bob', 20, 'IT')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [316]:
# Данные: (Отдел, Зарплата, Имя)
employees = [("IT", 1000, "A"), ("IT", 1500, "B"), ("HR", 800, "C")]

def RECORDREADER():
    return [(None, e) for e in employees]

def MAP(_, row):
    yield (row[0], row[1])

def REDUCE(dept, salaries):
    # Агрегация: SUM
    total = sum(salaries)
    yield (dept, total)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[('IT', 2500), ('HR', 800)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [317]:
import numpy as np


M = np.array([[1, 2], [3, 4]])
V = np.array([10, 20])

def RECORDREADER():
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            yield (None, ('M', i, j, M[i, j]))

    for j in range(len(V)):
        yield (None, ('V', j, V[j]))


def MAP_join(k, v):
    if v[0] == 'M':
        yield (v[2], ('M', v[1], v[3]))
    else:
        yield (v[1], ('V', v[2]))

def REDUCE_join(j, values):
    m_parts = []
    v_j = None
    for val in values:
        if val[0] == 'M': m_parts.append((val[1], val[2]))
        else: v_j = val[1]

    if v_j is not None:
        for i, m_ij in m_parts:
            yield (i, m_ij * v_j)


partial_products = list(MapReduce(RECORDREADER, MAP_join, REDUCE_join))


def MAP_sum(i, prod):
    yield (i, prod)

def REDUCE_sum(i, prods):
    yield (i, sum(prods))


def RECORDREADER_2():
    return partial_products

result = list(MapReduce(RECORDREADER_2, MAP_sum, REDUCE_sum))
print("Matrix-Vector Result:", result)

Matrix-Vector Result: [(0, np.int64(50)), (1, np.int64(110))]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [318]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [319]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), w * small_mat[i][j])

def REDUCE(key, values):
  (i, k) = key
  el_value = 0
  for v in values:
    el_value += v
  yield ((i, k), el_value)

Проверьте своё решение

In [320]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [321]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [322]:
import numpy as np

# Определение матриц
M = np.random.rand(2, 3)
N = np.random.rand(3, 4)

def RECORDREADER():
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            yield (None, ('M', i, j, M[i, j]))
    for j in range(N.shape[0]):
        for k in range(N.shape[1]):
            yield (None, ('N', j, k, N[j, k]))

def MAP(_, val):
    if val[0] == 'M':
        _, i, j, v = val
        yield (j, ('M', i, v))
    else:
        _, j, k, w = val
        yield (j, ('N', k, w))

def REDUCE(j, values):
    m_list = [v for v in values if v[0] == 'M']
    n_list = [v for v in values if v[0] == 'N']

    for _, i, v in m_list:
        for _, k, w in n_list:
            yield ((i, k), v * w)


def MAP_SUM(key, val):
    yield (key, val)

def REDUCE_SUM(key, values):
    yield (key, sum(values))


step1 = list(MapReduce(RECORDREADER, MAP, REDUCE))
def RECORDREADER_STEP2(): return step1
final_output = list(MapReduce(RECORDREADER_STEP2, MAP_SUM, REDUCE_SUM))


reference = np.matmul(M, N)
print(np.allclose(reference, asmatrix(final_output)))

True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [323]:
rows_a, common_dim, cols_b = 4, 10, 30
M = np.random.rand(rows_a, common_dim)
N = np.random.rand(common_dim, cols_b)
expected_matrix = np.matmul(M, N)


def INPUTFORMAT_DIST():
    yield [((0, i, j), M[i, j]) for i in range(M.shape[0]) for j in range(M.shape[1])]

    yield [((1, j, k), N[j, k]) for j in range(N.shape[0]) for k in range(N.shape[1])]

def MAP_JOIN(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val)) # j - ключ
    else:
        yield (idx1, (src, idx2, val)) # j - ключ

def REDUCE_JOIN(key, values):
    m_part = [v for v in values if v[0] == 0]
    n_part = [v for v in values if v[0] == 1]
    for m in m_part:
        for n in n_part:
            yield ((m[1], n[1]), m[2] * n[2])


dist_step1 = MapReduceDistributed(INPUTFORMAT_DIST, MAP_JOIN, REDUCE_JOIN)


step1_output = []
for pid, partition in dist_step1:
    step1_output.extend(list(partition))

def INPUTFORMAT_SUM():
    yield step1_output

def MAP_SUM(key, val):
    yield (key, val)

def REDUCE_SUM(key, values):
    yield (key, sum(values))

dist_step2 = MapReduceDistributed(INPUTFORMAT_SUM, MAP_SUM, REDUCE_SUM)

final_output = []
for pid, partition in dist_step2:
    final_output.extend(list(partition))

print(np.allclose(expected_matrix, asmatrix(final_output)))

340 key-value pairs were sent over a network.
1200 key-value pairs were sent over a network.
True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [324]:
def INPUTFORMAT_MULTIPLE():
    for i in range(M.shape[0]):
        yield [((0, i, j), M[i, j]) for j in range(M.shape[1])]

    mid = N.shape[1] // 2
    yield [((1, j, k), N[j, k]) for j in range(N.shape[0]) for k in range(0, mid)]
    yield [((1, j, k), N[j, k]) for j in range(N.shape[0]) for k in range(mid, N.shape[1])]

multi_step1 = MapReduceDistributed(INPUTFORMAT_MULTIPLE, MAP_JOIN, REDUCE_JOIN)

multi_step1_data = []
for _, partition in multi_step1:
    multi_step1_data.extend(list(partition))

def INPUTFORMAT_MULTI_SUM():
    yield multi_step1_data

multi_step2 = MapReduceDistributed(INPUTFORMAT_MULTI_SUM, MAP_SUM, REDUCE_SUM)

multi_res = []
for _, partition in multi_step2:
    multi_res.extend(list(partition))

print(np.allclose(expected_matrix, asmatrix(multi_res)))

340 key-value pairs were sent over a network.
1200 key-value pairs were sent over a network.
True
